In [49]:
import numpy as np
import pandas as pd
import tensorflow as tf
import joblib
from statsmodels.stats.proportion import proportions_ztest, confint_proportions_2indep

In [50]:
npz = np.load('Audiobooks_data_train.npz')
train_inputs = npz['inputs'].astype(float)
train_targets = npz['outputs'].astype(int)

npz = np.load('Audiobooks_data_validation.npz')
validation_inputs = npz['inputs'].astype(float)
validation_targets = npz['outputs'].astype(int)

npz = np.load('Audiobooks_data_test.npz')
test_inputs = npz['inputs'].astype(float)
test_targets = npz['outputs'].astype(int)

In [51]:
model = joblib.load("model.pkl")
y_score = model.predict(test_inputs)[:,1]

df = pd.DataFrame({
    'y_true': test_targets,                 
    'score': y_score      
})
df = df.sort_values(by='score', ascending=False).reset_index(drop=True)

14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step 


In [52]:
# Define treatment and control groups
k = 0.40

n_target = int(len(df) * k)

treatment = df.iloc[:n_target].copy() # Treatment group = top scored users according to the model
control = df.sample(n=n_target, random_state=42).copy() # Control group = random sample of same size

print(treatment.shape)
print(control.shape)
print(treatment['y_true'].sum())

(179, 2)
(179, 2)
155


In [53]:
#compute convertion rate
treatment_cr = treatment['y_true'].mean()
control_cr = control['y_true'].mean()

print(f"Treatment conversion rate = {round(treatment_cr, 4)}")
print(f"Control conversion rate = {round(control_cr, 4)}")


Treatment conversion rate = 0.8659
Control conversion rate = 0.486


In [54]:
# Compute absolute uplift and relative lift

absolute_uplift = treatment_cr - control_cr

if control_cr > 0:
    relative_lift = absolute_uplift / control_cr
else:
    relative_lift = np.nan

print(f"Absolute uplift = {round(absolute_uplift, 4)}")
print(f"Relative uplift = {round(relative_lift, 4) if not np.isnan(relative_lift) else 'Unknown'}")


Absolute uplift = 0.3799
Relative uplift = 0.7816


In [55]:
# Compute actual conversion counts
treatment_conversions = treatment['y_true'].sum()
control_conversions = control['y_true'].sum()

treatment_n = len(treatment)
control_n = len(control)

print(f"Treatment conversions = {treatment_conversions} out of {treatment_n}")
print(f"Control conversions = {control_conversions} out of {control_n}")

Treatment conversions = 155 out of 179
Control conversions = 87 out of 179


In [56]:
# Statistical significance test
# Using a two-proportion z-test because conversion is binary (0/1)
# Null hypothesis: treatment conversion <= control conversion
# Alternative hypothesis: treatment conversion > control conversion

count = np.array([treatment_conversions, control_conversions])
nobs = np.array([treatment_n, control_n])

z_stat, p_value = proportions_ztest(count=count, nobs=nobs, alternative='larger')

print(f"Z-statistic = {round(z_stat, 4)}")
print(f"P-value = {round(p_value, 4)}")

Z-statistic = 7.6792
P-value = 0.0


In [57]:
# Confidence interval for difference in conversion rates

ci_low, ci_high = confint_proportions_2indep(
    count1=treatment_conversions, nobs1=treatment_n,
    count2=control_conversions, nobs2=control_n,
    method='wald'
)

print(f"Lower bound (95% CI) {round(ci_low, 4)}")
print(f"Upper bound (95% CI) {round(ci_high, 4)}")


Lower bound (95% CI) 0.2913
Upper bound (95% CI) 0.4685


In [58]:
# Business impact
# Assumptions:
price = 15 #average revenue from one audiobook purchase
contact_cost = 1 #cost to target one user (email, ads, discount, etc.)

treatment_revenue = treatment_conversions * price 
control_revenue = control_conversions * price

treatment_cost = treatment_n * contact_cost
control_cost = control_n * contact_cost

treatment_profit = treatment_revenue - treatment_cost
control_profit = control_revenue - control_cost

incremental_profit = treatment_profit - control_profit

print("Profit Analysis")
print("Treatment revenue:", treatment_revenue)
print("Control revenue:", control_revenue)
print("Treatment profit:", treatment_profit)
print("Control profit:", control_profit)
print("Incremental profit from model targeting:", incremental_profit)


Profit Analysis
Treatment revenue: 2325
Control revenue: 1305
Treatment profit: 2146
Control profit: 1126
Incremental profit from model targeting: 1020


In [59]:
# Summary table
summary = pd.DataFrame({
    'Group': ['Treatment (Model Top-K)', 'Control (Random)'],
    'Users': [treatment_n, control_n],
    'Conversions': [treatment_conversions, control_conversions],
    'Conversion Rate': [treatment_cr, control_cr],
    'Revenue': [treatment_revenue, control_revenue],
    'Cost': [treatment_cost, control_cost],
    'Profit': [treatment_profit, control_profit]
})

print(summary)

                     Group  Users  Conversions  Conversion Rate  Revenue  \
0  Treatment (Model Top-K)    179          155         0.865922     2325   
1         Control (Random)    179           87         0.486034     1305   

   Cost  Profit  
0   179    2146  
1   179    1126  


In [60]:
import numpy as np
import pandas as pd
from statsmodels.stats.proportion import proportions_ztest, confint_proportions_2indep


# Sort users from highest probability to lowest probability
df = df.sort_values(by='score', ascending=False).reset_index(drop=True)

# ---------------------------------------------------------
# STEP 2: Choose the percentage of users you want to target
# ---------------------------------------------------------
# Example: target top 20% of users
k = 0.40

# Number of users to include in each group
n_target = int(len(df) * k)

print("Total users in test set:", len(df))
print("Users in each group:", n_target)

# ---------------------------------------------------------
# STEP 3: Define treatment and control groups
# ---------------------------------------------------------
# Treatment group = top scored users according to the model
treatment = df.iloc[:n_target].copy()

# Control group = random sample of same size
# random_state makes the result reproducible
control = df.sample(n=n_target, random_state=42).copy()

# ---------------------------------------------------------
# STEP 4: Compute conversion rates
# ---------------------------------------------------------
# Conversion rate = proportion of users who actually purchased
treatment_cr = treatment['y_true'].mean()
control_cr = control['y_true'].mean()

print("\n--- Conversion Rates ---")
print("Treatment conversion rate:", round(treatment_cr, 4))
print("Control conversion rate:", round(control_cr, 4))

# ---------------------------------------------------------
# STEP 5: Compute uplift / lift
# ---------------------------------------------------------
# Absolute uplift = direct difference in conversion rates
absolute_uplift = treatment_cr - control_cr

# Relative lift = how much better treatment is relative to control
# Protect against divide-by-zero
if control_cr > 0:
    relative_lift = absolute_uplift / control_cr
else:
    relative_lift = np.nan

print("\n--- Uplift ---")
print("Absolute uplift:", round(absolute_uplift, 4))
print("Relative lift:", round(relative_lift, 4) if not np.isnan(relative_lift) else "undefined")

# ---------------------------------------------------------
# STEP 6: Compute actual conversion counts
# ---------------------------------------------------------
treatment_conversions = treatment['y_true'].sum()
control_conversions = control['y_true'].sum()

treatment_n = len(treatment)
control_n = len(control)

print("\n--- Conversion Counts ---")
print("Treatment conversions:", treatment_conversions, "out of", treatment_n)
print("Control conversions:", control_conversions, "out of", control_n)

# ---------------------------------------------------------
# STEP 7: Statistical significance test
# ---------------------------------------------------------
# We use a two-proportion z-test because conversion is binary (0/1)
# Null hypothesis: treatment conversion <= control conversion
# Alternative hypothesis: treatment conversion > control conversion

count = np.array([treatment_conversions, control_conversions])
nobs = np.array([treatment_n, control_n])

z_stat, p_value = proportions_ztest(count=count, nobs=nobs, alternative='larger')

print("\n--- Two-Proportion Z-Test ---")
print("Z-statistic:", round(z_stat, 4))
print("P-value:", round(p_value, 6))

# ---------------------------------------------------------
# STEP 8: Confidence interval for difference in conversion rates
# ---------------------------------------------------------
# This estimates a likely range for (treatment rate - control rate)

ci_low, ci_high = confint_proportions_2indep(
    count1=treatment_conversions, nobs1=treatment_n,
    count2=control_conversions, nobs2=control_n,
    method='wald'
)

print("\n--- 95% Confidence Interval for Conversion Difference ---")
print("Lower bound:", round(ci_low, 4))
print("Upper bound:", round(ci_high, 4))

# ---------------------------------------------------------
# STEP 9: Business impact calculation
# ---------------------------------------------------------
# Assumptions:
# price = average revenue from one audiobook purchase
# contact_cost = cost to target one user (email, ads, discount, etc.)

price = 15          # example average revenue per purchase
contact_cost = 1    # example cost per targeted user

# Total revenue from each group
treatment_revenue = treatment_conversions * price
control_revenue = control_conversions * price

# Total targeting cost
treatment_cost = treatment_n * contact_cost
control_cost = control_n * contact_cost

# Profit = revenue - cost
treatment_profit = treatment_revenue - treatment_cost
control_profit = control_revenue - control_cost

incremental_profit = treatment_profit - control_profit

print("\n--- Revenue / Profit Analysis ---")
print("Treatment revenue:", treatment_revenue)
print("Control revenue:", control_revenue)
print("Treatment profit:", treatment_profit)
print("Control profit:", control_profit)
print("Incremental profit from model targeting:", incremental_profit)

# ---------------------------------------------------------
# STEP 10: Summary table
# ---------------------------------------------------------
summary = pd.DataFrame({
    'Group': ['Treatment (Model Top-K)', 'Control (Random)'],
    'Users': [treatment_n, control_n],
    'Conversions': [treatment_conversions, control_conversions],
    'Conversion Rate': [treatment_cr, control_cr],
    'Revenue': [treatment_revenue, control_revenue],
    'Cost': [treatment_cost, control_cost],
    'Profit': [treatment_profit, control_profit]
})

print("\n--- Summary Table ---")
print(summary)

Total users in test set: 448
Users in each group: 179

--- Conversion Rates ---
Treatment conversion rate: 0.8659
Control conversion rate: 0.4916

--- Uplift ---
Absolute uplift: 0.3743
Relative lift: 0.7614

--- Conversion Counts ---
Treatment conversions: 155 out of 179
Control conversions: 88 out of 179

--- Two-Proportion Z-Test ---
Z-statistic: 7.5834
P-value: 0.0

--- 95% Confidence Interval for Conversion Difference ---
Lower bound: 0.2857
Upper bound: 0.4629

--- Revenue / Profit Analysis ---
Treatment revenue: 2325
Control revenue: 1320
Treatment profit: 2146
Control profit: 1141
Incremental profit from model targeting: 1005

--- Summary Table ---
                     Group  Users  Conversions  Conversion Rate  Revenue  \
0  Treatment (Model Top-K)    179          155         0.865922     2325   
1         Control (Random)    179           88         0.491620     1320   

   Cost  Profit  
0   179    2146  
1   179    1141  


In [71]:
import streamlit as st
import matplotlib.pyplot as plt

# -----------------------------
# Metrics
# -----------------------------
st.title("Audiobook Purchase Dashboard")

st.metric("Treatment Conversion", "86.6%")
st.metric("Control Conversion", "49.2%")
st.metric("Incremental Profit", "$1005")

# -----------------------------
# 1. Conversion Rate Plot
# -----------------------------
fig_conversion = plt.figure()
groups = ['Control', 'Treatment']
rates = [0.4916, 0.8659]

plt.bar(groups, rates)
plt.title("Conversion Rate Comparison")
plt.ylabel("Conversion Rate")

st.pyplot(fig_conversion)

# -----------------------------
# 2. Gain Chart (simple version)
# -----------------------------
fig_gain = plt.figure()

# example curve (you can replace with real one later)
x = [0, 0.2, 0.4, 0.6, 0.8, 1]
y_model = [0, 0.35, 0.55, 0.7, 0.85, 1]
y_random = [0, 0.2, 0.4, 0.6, 0.8, 1]

plt.plot(x, y_model, label="Model population")
plt.plot(x, y_random, linestyle='--', label="Random population")

plt.title("Gain Chart")
plt.xlabel("Population %")
plt.ylabel("Captured Conversions %")
plt.legend()
st.pyplot(fig_gain)

# -----------------------------
# 3. Profit Comparison
# -----------------------------
fig_profit = plt.figure()

profits = [1141, 2146]  # control, treatment
groups = ['Control', 'Treatment']

plt.bar(groups, profits)
plt.title("Profit Comparison")
plt.ylabel("Profit ($)")

st.pyplot(fig_profit)

2026-03-24 01:00:42.842 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-24 01:00:42.843 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-24 01:00:42.843 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-24 01:00:42.843 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-24 01:00:42.844 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-24 01:00:42.844 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-24 01:00:42.844 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-24 01:00:42.845 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar

DeltaGenerator()

In [69]:
import streamlit as st

st.title("Audiobook Purchase Targeting Dashboard")

st.metric("Treatment Conversion", "86.6%")
st.metric("Control Conversion", "49.2%")
st.metric("Incremental Profit", "$1005")

st.pyplot(fig_conversion)
st.pyplot(fig_gain)
st.pyplot(fig_profit)

2026-03-24 01:00:21.505 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-24 01:00:21.506 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-24 01:00:21.506 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-24 01:00:21.507 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-24 01:00:21.507 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-24 01:00:21.507 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-24 01:00:21.507 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-24 01:00:21.508 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar

DeltaGenerator()